In [ ]:
# Install deps
import subprocess
subprocess.run(['pip','install','-q','transformers','peft','datasets',
                'accelerate','bitsandbytes','llama-cpp-python'], check=True)

In [ ]:
import json, random
from pathlib import Path
from datasets import Dataset

# Load dataset dari Kaggle input
DATASET_PATH = Path('/kaggle/input/aland-ai-dataset/dataset.jsonl')
data = []
with open(DATASET_PATH, encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            if len(obj.get('messages', [])) >= 2:
                data.append(obj)
        except: pass

random.shuffle(data)
data = data[:60000]  # Kaggle P100 bisa handle lebih banyak
print(f'Dataset: {len(data)} pasangan')
dataset = Dataset.from_list(data)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                              device_map='auto', trust_remote_code=True)
print('Model loaded ✅')

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM'
))
model.print_trainable_parameters()

In [ ]:
def fmt(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    tok = tokenizer(text, truncation=True, max_length=512, padding='max_length')
    tok['labels'] = tok['input_ids'].copy()
    return tok

tokenized = dataset.map(fmt, remove_columns=['messages'])

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir='/kaggle/working/lora',
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4, fp16=True,
        logging_steps=100, save_steps=1000,
        warmup_ratio=0.05, report_to='none',
    ),
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()
print('Training selesai ✅')

In [ ]:
# Merge LoRA + export GGUF
from peft import PeftModel
import subprocess

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, '/kaggle/working/lora').merge_and_unload()
merged.save_pretrained('/kaggle/working/merged')
tokenizer.save_pretrained('/kaggle/working/merged')

subprocess.run(['git','clone','-q','https://github.com/ggerganov/llama.cpp','/kaggle/working/llama.cpp'], check=True)
subprocess.run(['pip','install','-q','gguf'], check=True)
subprocess.run([
    'python', '/kaggle/working/llama.cpp/convert_hf_to_gguf.py',
    '/kaggle/working/merged',
    '--outfile', '/kaggle/working/aland-id-q4.gguf',
    '--outtype', 'q4_k_m'
], check=True)
print('GGUF export selesai ✅')
print('Output: /kaggle/working/aland-id-q4.gguf')